# Agrupamiento de estaciones y respuesta a P3 — MiBici

**Proyecto:** TC5061 · Equipo 52 · Pregunta P3 (agrupamiento de estaciones)
**Autor:** Joel Sánchez (PittJR1607) · **Rama:** `feature/p3-clustering`
**Antecedente:** `3.0-js-p3-exploracion-estaciones.ipynb` (auditoría de datos, viajes redondos, cohortes de antigüedad y construcción de las 42 características en `features.py`).

**Propósito.** Evaluar la línea base geográfica y los modelos candidatos de P3, verificar la sensibilidad del agrupamiento frente a los **viajes redondos (`Origen_Id == Destino_Id`)** y a las **estaciones incorporadas en los últimos años (expansión 2026)**, validar su estabilidad temporal entre eras y responder con cifras y mapas la pregunta de negocio:
> **P3:** *¿Qué grupos de estaciones comparten patrones de uso y dónde se ubican en la ciudad?*

La lógica reproducible y el registro en MLflow viven en `equipo52_module/modeling/train_p3.py`; este notebook ejecuta y analiza en detalle los mismos modelos sobre la matriz de **469 estaciones × 42 características**.

In [1]:
# Objetivo: cargar las librerías, reconstruir la matriz de estaciones desde features.py y preparar
# las matrices estandarizadas para el análisis.
# Resultado esperado: confirmación de las 469 estaciones activas y 42 características estandarizadas.

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import HDBSCAN, AgglomerativeClustering, KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path.cwd().parent))
from equipo52_module.config import (
    DURACION_MINIMA_SEGUNDOS,
    ESTACIONES_EXCLUIDAS,
    FECHA_FIN,
    NOMENCLATURA_FILE,
    SEED,
    VIAJES_FILE,
)
from equipo52_module.features import (
    SEGUNDOS_REDONDO_CORTO,
    construir_matriz_p3,
    quitar_viajes_cortos,
    quitar_viajes_redondos_cortos,
)
from equipo52_module.modeling.train_p3 import (
    CORTE_ERAS,
    GEO,
    K_GEO,
    MESES_ESTACION_CONSOLIDADA,
    N_INIT,
    cargar_estaciones,
    describir_grupos,
    estabilidad_bootstrap,
    estabilidad_entre_eras,
)

pd.set_option("display.max_columns", 60)

viajes = quitar_viajes_cortos(pd.read_parquet(VIAJES_FILE), DURACION_MINIMA_SEGUNDOS)
estaciones = cargar_estaciones(NOMENCLATURA_FILE, ESTACIONES_EXCLUIDAS)
matriz = construir_matriz_p3(viajes, estaciones, pd.Timestamp(FECHA_FIN))
columnas = [c for c in matriz.columns if c not in GEO]
escalador = StandardScaler().fit(matriz[columnas])
X = pd.DataFrame(escalador.transform(matriz[columnas]), columns=columnas, index=matriz.index)

print(f"Matriz lista: {X.shape[0]} estaciones × {X.shape[1]} características de uso estandarizadas")

2026-10-18 05:49:02.067 | INFO     | equipo52_module.config:<module>:11 - PROJ_ROOT path is: /Users/pittjr/JSO_DEV/ITESM/MAESTRIA_IA/3rdTrim/TC5061-Proyecto-Equipo52


/Users/pittjr/JSO_DEV/ITESM/MAESTRIA_IA/3rdTrim/TC5061-Proyecto-Equipo52/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Matriz lista: 469 estaciones × 42 características de uso estandarizadas


## 1. Línea base: agrupar únicamente por cercanía geográfica (`geo`, $k = 5$)

Un planeador urbano sin un modelo de Machine Learning dividiría la red trazando zonas geográficas en el mapa (por ejemplo, 5 sectores: centro, poniente, oriente, norte/Zapopan y sur/Tlaquepaque). Implementamos esa solución ingenua aplicando K-Means ($k = 5$) exclusivamente sobre `latitud` y `longitud`, y medimos sus métricas internas **en el espacio de las 42 características de uso**: si las estaciones cercanas se usaran igual, esta partición tendría buena silueta de uso.

In [2]:
# Objetivo: ajustar la línea base geográfica (K-Means k = 5 sobre latitud y longitud) y medir su
# coherencia en el espacio de uso (42 características).
# Resultado esperado: silueta, Davies-Bouldin y Calinski-Harabasz en el espacio de uso y mapa de las 5 zonas.

X_geo = pd.DataFrame(StandardScaler().fit_transform(matriz[GEO]), columns=GEO, index=matriz.index)
modelo_geo = KMeans(n_clusters=K_GEO, n_init=N_INIT, random_state=SEED).fit(X_geo)

sil_geo = silhouette_score(X, modelo_geo.labels_)
db_geo = davies_bouldin_score(X, modelo_geo.labels_)
ch_geo = calinski_harabasz_score(X, modelo_geo.labels_)

print(f"Línea base geográfica (k = {K_GEO}) evaluada en el espacio de uso:")
print(f"  hist_silhouette:        {sil_geo:.3f}")
print(f"  hist_davies_bouldin:    {db_geo:.3f}")
print(f"  hist_calinski_harabasz: {ch_geo:.1f}\n")

display(describir_grupos(matriz, modelo_geo.labels_, estaciones)[
    ["estaciones", "est_GDL", "est_ZPN", "est_TLQ", "salidas_por_mes", "balance", "pico_manana_07_09", "pico_tarde_17_19", "fin_de_semana"]
])

fig, ax = plt.subplots(figsize=(7.5, 6.5))
pts = ax.scatter(matriz["longitud"], matriz["latitud"], c=modelo_geo.labels_, cmap="tab10", s=26, alpha=0.85)
ax.set_title(f"Línea base geográfica (k = {K_GEO}): silueta en espacio de uso = {sil_geo:.3f}")
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.legend(*pts.legend_elements(), title="Zona geo")
fig.tight_layout()

Línea base geográfica (k = 5) evaluada en el espacio de uso:
  hist_silhouette:        0.019
  hist_davies_bouldin:    3.267
  hist_calinski_harabasz: 20.9



,estaciones,est_GDL,est_ZPN,est_TLQ,salidas_por_mes,balance,pico_manana_07_09,pico_tarde_17_19,fin_de_semana
grupo,,,,,,,,,
0,49,1,48,0,386.982,0.008,0.216,0.225,0.181
1,142,142,0,0,1274.994,-0.033,0.164,0.258,0.194
2,101,101,0,0,940.072,0.078,0.234,0.208,0.194
3,93,78,0,15,317.575,-0.018,0.229,0.185,0.254
4,84,30,54,0,308.074,-0.124,0.189,0.276,0.219


### Conclusión de la línea base geográfica

La silueta en el espacio de uso es de apenas **0.019** (prácticamente cero, con un Davies-Bouldin alto de **3.267** y Calinski-Harabasz de **20.9**). En una misma cuadra o polígono geográfico conviven estaciones de alta rotación sobre avenidas principales con estaciones secundarias de perfil residencial o recreativo; por tanto, **dividir el mapa por cercanía física no separa los patrones de operación**.

---

## 2. Selección del número de grupos ($k$) en K-Means y agrupamiento jerárquico Ward

Recorremos $k = 2 \dots 12$ sobre las 42 características estandarizadas comparando las tres métricas internas (`silhouette`, `davies_bouldin` y `calinski_harabasz`) tanto en K-Means como en el agrupamiento jerárquico aglomerativo con enlace **Ward**.

In [3]:
# Objetivo: barrer k de 2 a 12 para K-Means y Ward registrando las tres métricas internas.
# Resultado esperado: tabla comparativa por k y gráficas de silueta y Calinski-Harabasz.

filas_k = []
for k in range(2, 13):
    et_km = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED).fit_predict(X)
    et_ward = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X)
    filas_k.append({
        "k": k,
        "kmeans_silhouette": round(silhouette_score(X, et_km), 3),
        "kmeans_davies_bouldin": round(davies_bouldin_score(X, et_km), 3),
        "kmeans_calinski_harabasz": round(calinski_harabasz_score(X, et_km), 1),
        "ward_silhouette": round(silhouette_score(X, et_ward), 3),
        "ward_davies_bouldin": round(davies_bouldin_score(X, et_ward), 3),
        "condordancia_ari": round(adjusted_rand_score(et_km, et_ward), 3),
    })
tabla_barrido = pd.DataFrame(filas_k).set_index("k")
display(tabla_barrido)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))
ax1.plot(tabla_barrido.index, tabla_barrido["kmeans_silhouette"], marker="o", label="K-Means")
ax1.plot(tabla_barrido.index, tabla_barrido["ward_silhouette"], marker="s", label="Ward")
ax1.axhline(sil_geo, color="#d62728", ls=":", label=f"Baseline geo ({sil_geo:.3f})")
ax1.axvline(2, color="gray", ls="--", lw=1, label="Máximo global (k = 2)")
ax1.axvline(4, color="#2ca02c", ls="--", lw=1, label="Subdivisión operativa (k = 4)")
ax1.set_title("Coeficiente de silueta por número de grupos (k)")
ax1.set_xlabel("Número de grupos (k)")
ax1.set_ylabel("Silueta")
ax1.set_xticks(range(2, 13))
ax1.legend(fontsize=8)

ax2.plot(tabla_barrido.index, tabla_barrido["kmeans_davies_bouldin"], marker="o", color="#ff7f0e", label="Davies-Bouldin K-Means (menor es mejor)")
ax2.set_title("Índice de Davies-Bouldin por k")
ax2.set_xlabel("Número de grupos (k)")
ax2.set_ylabel("Davies-Bouldin")
ax2.set_xticks(range(2, 13))
ax2.legend(fontsize=8)
fig.tight_layout()

,kmeans_silhouette,kmeans_davies_bouldin,kmeans_calinski_harabasz,ward_silhouette,ward_davies_bouldin,condordancia_ari
k,,,,,,
2,0.151,2.309,75.2,0.129,2.541,0.565
3,0.092,2.349,56.0,0.089,2.604,0.307
4,0.105,2.309,50.6,0.044,2.541,0.459
5,0.064,2.330,43.6,0.057,2.631,0.417
6,0.065,2.330,39.3,0.054,2.495,0.413
7,0.071,2.463,35.6,0.058,2.578,0.440
8,0.087,2.310,33.4,0.062,2.397,0.372
9,0.073,2.222,31.5,0.066,2.414,0.352
10,0.053,1.818,29.5,0.052,2.306,0.370


### Interpretación del barrido de $k$: por qué conservamos $k = 2$ y $k = 4$

- **Óptimo global en $k = 2$ (`hist_silhouette = 0.151`, 8 veces mayor que el baseline geográfico de `0.019`; `Davies-Bouldin = 2.309`; `Calinski-Harabasz = 75.2`):** el corte más fuerte de toda la red separa a las **270 estaciones consolidadas de alto flujo** de las **199 estaciones de menor intensidad y expansión reciente**.
- **Segundo punto de inflexión en $k = 4$ (`hist_silhouette = 0.105`, `Davies-Bouldin = 2.309`, `Calinski-Harabasz = 50.6`):** al pasar de $k = 3$ (silueta 0.092) a $k = 4$ (0.105) la silueta repunta y el Davies-Bouldin iguala el nivel de $k = 2$ (**2.309**), antes de caer a 0.064 en $k = 5$. Mientras que $k = 2$ solo distingue "grandes consolidadas vs. chicas/recientes", **$k = 4$ separa la dirección del flujo y el horario pico** (núcleo central, atractoras vespertinas, emisoras matutinas y periféricas recreativas), que es el nivel accionable para planeación urbana y rebalanceo. Por eso registramos ambas corridas (`p3-kmeans-k2` y `p3-kmeans-k4`) en MLflow.

---

## 3. Pruebas de sensibilidad a las recomendaciones del equipo

Antes de interpretar los grupos finales verificamos cuantitativamente las dos inquietudes planteadas en equipo:
1. **¿Cambian los grupos si quitamos los viajes con `Origen_Id == Destino_Id`?** Probamos: (a) matriz base (con viajes redondos $\ge 60$ s), (b) quitando los viajes redondos cortos de **1 a 3 minutos** (`quitar_viajes_redondos_cortos`, posibles devoluciones tardías por falla mecánica) y (c) quitando **todos** los viajes `Origen_Id == Destino_Id`.
2. **¿Perturban las 98 estaciones abiertas en 2026 ($\le 6$ meses) o las variables de antigüedad a los grupos?** Probamos: (a) ajustar K-Means con las 469 estaciones, (b) ajustar el escalador y los centroides **únicamente con las 371 estaciones consolidadas ($> 6$ meses)** y proyectar las 98 nuevas (`--solo-consolidadas`), y (c) agrupar **sin las 3 variables de evolución** (`meses_activa`, `share_viajes_ultimos_12m`, `crecimiento_vs_esperado`), usando solo las **39 variables de comportamiento puro** (actividad mensual, firma horaria, firma semanal, balance y tipo de viaje).

In [4]:
# Objetivo: ejecutar las pruebas de sensibilidad para viajes redondos (Origen == Destino)
# y para las estaciones nuevas de 2026 (<= 6 meses).
# Resultado esperado: tabla con silueta, Davies-Bouldin y concordancia ARI contra el modelo base en k = 2 y k = 4.

fin = pd.Timestamp(FECHA_FIN)

# Variante 1: sin viajes redondos cortos (1 a 3 min)
viajes_sin_rc = quitar_viajes_redondos_cortos(viajes, SEGUNDOS_REDONDO_CORTO)
matriz_sin_rc = construir_matriz_p3(viajes_sin_rc, estaciones, fin)
X_sin_rc = pd.DataFrame(StandardScaler().fit_transform(matriz_sin_rc[columnas]), columns=columnas, index=matriz_sin_rc.index)

# Variante 2: sin ningún viaje Origen == Destino
viajes_sin_red = viajes[viajes["Origen_Id"] != viajes["Destino_Id"]].reset_index(drop=True)
matriz_sin_red = construir_matriz_p3(viajes_sin_red, estaciones, fin)
cols_sin_red = [c for c in columnas if c != "prop_viajes_redondos"]
X_sin_red = pd.DataFrame(StandardScaler().fit_transform(matriz_sin_red[cols_sin_red]), columns=cols_sin_red, index=matriz_sin_red.index)

# Variante 3: entrenando solo con las 371 estaciones consolidadas (> 6 meses)
mascara_con = matriz["meses_activa"] > MESES_ESTACION_CONSOLIDADA
esc_con = StandardScaler().fit(matriz.loc[mascara_con, columnas])
X_con_fit = pd.DataFrame(esc_con.transform(matriz.loc[mascara_con, columnas]), columns=columnas, index=matriz.index[mascara_con])
X_con_all = pd.DataFrame(esc_con.transform(matriz[columnas]), columns=columnas, index=matriz.index)

# Variante 4: sin variables de evolución (39 variables de comportamiento puro)
cols_puro = [c for c in columnas if c not in ("meses_activa", "share_viajes_ultimos_12m", "crecimiento_vs_esperado")]
X_puro = pd.DataFrame(StandardScaler().fit_transform(matriz[cols_puro]), columns=cols_puro, index=matriz.index)

resultados_sens = []
for k in [2, 4]:
    km_base = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED).fit(X)
    km_rc = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED).fit(X_sin_rc)
    km_sr = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED).fit(X_sin_red)
    km_con = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED).fit(X_con_fit)
    pred_con_all = km_con.predict(X_con_all)
    km_puro = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED).fit(X_puro)

    resultados_sens.extend([
        {"k": k, "Variante": "0. Base (469 est., 42 vars, redondos >= 60 s)", "Estaciones ajuste": len(X),
         "Silueta (ajuste)": round(silhouette_score(X, km_base.labels_), 3),
         "Davies-Bouldin": round(davies_bouldin_score(X, km_base.labels_), 3),
         "ARI vs. Base": 1.000},
        {"k": k, "Variante": "1. Sin viajes redondos cortos (1-3 min)", "Estaciones ajuste": len(X_sin_rc),
         "Silueta (ajuste)": round(silhouette_score(X_sin_rc, km_rc.labels_), 3),
         "Davies-Bouldin": round(davies_bouldin_score(X_sin_rc, km_rc.labels_), 3),
         "ARI vs. Base": round(adjusted_rand_score(km_base.labels_, km_rc.labels_), 3)},
        {"k": k, "Variante": "2. Sin ningún viaje Origen == Destino (41 vars)", "Estaciones ajuste": len(X_sin_red),
         "Silueta (ajuste)": round(silhouette_score(X_sin_red, km_sr.labels_), 3),
         "Davies-Bouldin": round(davies_bouldin_score(X_sin_red, km_sr.labels_), 3),
         "ARI vs. Base": round(adjusted_rand_score(km_base.labels_, km_sr.labels_), 3)},
        {"k": k, "Variante": "3. Ajuste solo en consolidadas (> 6 m, 371 est.)", "Estaciones ajuste": len(X_con_fit),
         "Silueta (ajuste)": round(silhouette_score(X_con_fit, km_con.labels_), 3),
         "Davies-Bouldin": round(davies_bouldin_score(X_con_fit, km_con.labels_), 3),
         "ARI vs. Base": round(adjusted_rand_score(km_base.labels_, pred_con_all), 3)},
        {"k": k, "Variante": "4. Uso puro sin antigüedad/evolución (39 vars)", "Estaciones ajuste": len(X_puro),
         "Silueta (ajuste)": round(silhouette_score(X_puro, km_puro.labels_), 3),
         "Davies-Bouldin": round(davies_bouldin_score(X_puro, km_puro.labels_), 3),
         "ARI vs. Base": round(adjusted_rand_score(km_base.labels_, km_puro.labels_), 3)},
    ])

display(pd.DataFrame(resultados_sens).set_index(["k", "Variante"]))

Estaciones ajuste  \
k Variante                                                              
2 0. Base (469 est., 42 vars, redondos >= 60 s)                   469   
  1. Sin viajes redondos cortos (1-3 min)                         469   
  2. Sin ningún viaje Origen == Destino (41 vars)                 469   
  3. Ajuste solo en consolidadas (> 6 m, 371 est.)                371   
  4. Uso puro sin antigüedad/evolución (39 vars)                  469   
4 0. Base (469 est., 42 vars, redondos >= 60 s)                   469   
  1. Sin viajes redondos cortos (1-3 min)                         469   
  2. Sin ningún viaje Origen == Destino (41 vars)                 469   
  3. Ajuste solo en consolidadas (> 6 m, 371 est.)                371   
  4. Uso puro sin antigüedad/evolución (39 vars)                  469   

                                                    Silueta (ajuste)  \
k Variante                                                             
2 0. Base (469 est., 42 vars, redondos >= 60 s)                0.151   
  1. Sin viajes redondos cortos (1-3 min)                      0.150   
  2. Sin ningún viaje Origen == Destino (41 vars)              0.149   
  3. Ajuste solo en consolidadas (> 6 m, 371 est.)             0.154   
  4. Uso puro sin antigüedad/evolución (39 vars)               0.149   
4 0. Base (469 est., 42 vars, redondos >= 60 s)                0.105   
  1. Sin viajes redondos cortos (1-3 min)                      0.104   
  2. Sin ningún viaje Origen == Destino (41 vars)              0.110   
  3. Ajuste solo en consolidadas (> 6 m, 371 est.)             0.127   
  4. Uso puro sin antigüedad/evolución (39 vars)               0.112   

                                                    Davies-Bouldin  \
k Variante                                                           
2 0. Base (469 est., 42 vars, redondos >= 60 s)              2.309   
  1. Sin viajes redondos cortos (1-3 min)                    2.315   
  2. Sin ningún viaje Origen == Destino (41 vars)            2.295   
  3. Ajuste solo en consolidadas (> 6 m, 371 est.)           2.204   
  4. Uso puro sin antigüedad/evolución (39 vars)             2.301   
4 0. Base (469 est., 42 vars, redondos >= 60 s)              2.309   
  1. Sin viajes redondos cortos (1-3 min)                    2.315   
  2. Sin ningún viaje Origen == Destino (41 vars)            2.224   
  3. Ajuste solo en consolidadas (> 6 m, 371 est.)           2.102   
  4. Uso puro sin antigüedad/evolución (39 vars)             2.218   

                                                    ARI vs. Base  
k Variante                                                        
2 0. Base (469 est., 42 vars, redondos >= 60 s)            1.000  
  1. Sin viajes redondos cortos (1-3 min)                  1.000  
  2. Sin ningún viaje Origen == Destino (41 vars)          0.958  
  3. Ajuste solo en consolidadas (> 6 m, 371 est.)         0.798  
  4. Uso puro sin antigüedad/evolución (39 vars)           0.805  
4 0. Base (469 est., 42 vars, redondos >= 60 s)            1.000  
  1. Sin viajes redondos cortos (1-3 min)                  0.924  
  2. Sin ningún viaje Origen == Destino (41 vars)          0.826  
  3. Ajuste solo en consolidadas (> 6 m, 371 est.)         0.641  
  4. Uso puro sin antigüedad/evolución (39 vars)           0.796

### Conclusiones de las pruebas de sensibilidad

1. **Los viajes redondos (`Origen_Id == Destino_Id`) no perturban el agrupamiento:**
   - Al descartar los 217,916 viajes redondos de **1 a 3 minutos** (posibles devoluciones tardías tras probar una bicicleta defectuosa), la asignación de estaciones es **idéntica en $k = 2$ (ARI = 1.000)** y casi idéntica en **$k = 4$ (ARI = 0.924)**.
   - Incluso si se eliminan por completo los 1.52 M de viajes `Origen_Id == Destino_Id`, el ARI sigue siendo **0.958 ($k = 2$)** y **0.826 ($k = 4$)**. Conservar los viajes redondos $\ge 60$ s es la decisión correcta porque no mete ruido y en cambio permite identificar a las estaciones con vocación recreativa y dominical.
2. **El efecto de las 98 estaciones abiertas en 2026 está controlado y cuantificado:**
   - Cuando ajustamos K-Means **únicamente sobre las 371 estaciones consolidadas ($> 6$ meses)** (`p3-kmeans-k4-consolidadas`), la cohesión interna mejora: en $k = 4$ la silueta dentro de las consolidadas sube de **0.105 a 0.127** y el Davies-Bouldin baja de **2.309 a 2.102** (en $k = 2$ la silueta sube de **0.151 a 0.154** y el DB baja a **2.204**).
   - Aún más importante: cuando quitamos por completo las 3 variables de antigüedad/evolución (`meses_activa`, `share_viajes_ultimos_12m`, `crecimiento_vs_esperado`) y agrupamos con las **39 variables de uso puro**, la silueta en $k = 4$ sube a **0.112** y la concordancia con la matriz completa es **ARI = 0.796** (**0.805** en $k = 2$). Es decir, **los grupos están determinados por su firma horaria, semanal y de flujo neto, no por un sesgo de antigüedad**.

---

## 4. Contraste estructural (Ward) y diagnóstico del continuo de estaciones (HDBSCAN)

Verificamos ahora qué ocurre al cambiar de familia de algoritmo:
- **Agrupamiento jerárquico aglomerativo (enlace Ward):** no depende de una inicialización aleatoria y permite inspeccionar en el dendrograma cómo se anidan los 4 grupos dentro de los 2 grandes regímenes.
- **HDBSCAN (basado en densidad):** busca islas densas separadas por zonas vacías y asigna etiqueta `-1` a las estaciones que no pertenecen a ningún núcleo denso.

In [5]:
# Objetivo: visualizar el dendrograma de Ward y ejecutar HDBSCAN con min_cluster_size = 15 y 5.
# Resultado esperado: dendrograma jerárquico y proporción de estaciones marcadas como ruido (-1) por HDBSCAN.

etiquetas_km2 = KMeans(n_clusters=2, n_init=N_INIT, random_state=SEED).fit_predict(X)
etiquetas_ward2 = AgglomerativeClustering(n_clusters=2, linkage="ward").fit_predict(X)
print(f"Ward k = 2 -> silueta: {silhouette_score(X, etiquetas_ward2):.3f} · ARI Ward vs. K-Means: {adjusted_rand_score(etiquetas_km2, etiquetas_ward2):.3f}")

for mcs in [15, 5]:
    et_hdb = HDBSCAN(min_cluster_size=mcs).fit_predict(X)
    nucleo = et_hdb >= 0
    n_grupos = int(et_hdb[nucleo].max() + 1) if nucleo.any() else 0
    sil_nuc = f"{silhouette_score(X[nucleo], et_hdb[nucleo]):.3f}" if n_grupos >= 2 else "n/a (< 2 grupos)"
    print(f"HDBSCAN (min_cluster_size = {mcs:2d}) -> grupos densos: {n_grupos} · estaciones en núcleo: {int(nucleo.sum()):3d} · ruido (-1): {(~nucleo).mean():.1%} ({int((~nucleo).sum())} est.) · silueta núcleo: {sil_nuc}")

fig, ax = plt.subplots(figsize=(12, 4.2))
dendrogram(linkage(X.to_numpy(), method="ward"), truncate_mode="lastp", p=30, ax=ax)
ax.set_title("Dendrograma Ward (truncado a 30 ramas): dos grandes ramas que se subdividen en cuatro grupos")
ax.set_ylabel("Distancia euclidiana (Ward)")
fig.tight_layout()

Ward k = 2 -> silueta: 0.129 · ARI Ward vs. K-Means: 0.565
HDBSCAN (min_cluster_size = 15) -> grupos densos: 0 · estaciones en núcleo:   0 · ruido (-1): 100.0% (469 est.) · silueta núcleo: n/a (< 2 grupos)
HDBSCAN (min_cluster_size =  5) -> grupos densos: 2 · estaciones en núcleo:  20 · ruido (-1): 95.7% (449 est.) · silueta núcleo: 0.376


/Users/pittjr/JSO_DEV/ITESM/MAESTRIA_IA/3rdTrim/TC5061-Proyecto-Equipo52/.venv/lib/python3.12/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/pittjr/JSO_DEV/ITESM/MAESTRIA_IA/3rdTrim/TC5061-Proyecto-Equipo52/.venv/lib/python3.12/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


### Qué nos dicen Ward y HDBSCAN sobre la naturaleza de los datos de MiBici

- **El dendrograma de Ward confirma la jerarquía $k = 2 \to k = 4$:** la red se parte primero en dos grandes ramas (concordancia con K-Means **ARI = 0.565**, silueta **0.129**) y cada rama se subdivide de forma natural en dos subgrupos para formar los **4 perfiles operativos**.
- **HDBSCAN demuestra por qué las siluetas de toda la red rondan 0.11–0.15:** con `min_cluster_size = 15` marca el **100 % de las estaciones como transición (`-1`)**, y al bajar a `min_cluster_size = 5` solo aísla **20 estaciones ultra-compactas** (con silueta interna alta de **0.376**) dejando el **95.7 % (449 estaciones)** como espectro continuo. En una red urbana real las estaciones no forman islas desconectadas: una estación residencial se vuelve gradualmente mixta y luego comercial conforme nos acercamos a los corredores de empleo. Por eso un método de partición (K-Means / Ward) es el adecuado para zonificar toda la red, mientras que HDBSCAN sirve como detector de estaciones extremas.

---

## 5. Validación de estabilidad temporal entre eras y por remuestreo (bootstrap)

Como en clustering no existe variable objetivo, validamos que los grupos no sean un accidente del corte de tiempo ni de unas cuantas estaciones:
1. **Estabilidad entre eras (`estabilidad_eras_ari`):** dividimos los 142 meses en **Era A (dic-2014 a ago-2020, 69 meses)** y **Era B (sep-2020 a sep-2026, 73 meses)**, recalculamos las 42 características de cada estación por separado en cada era y predecimos su grupo con el modelo ajustado.
2. **Estabilidad por bootstrap (`estabilidad_bootstrap_ari`):** re-entrenamos K-Means 30 veces tomando al azar el **80 % de las estaciones (375 de 469)** y medimos el ARI contra la asignación original.

In [6]:
# Objetivo: calcular la estabilidad temporal entre eras (2014-2020 vs. 2020-2026) y la estabilidad
# bootstrap (30 remuestreos al 80 %) para k = 2 y k = 4.
# Resultado esperado: ARI entre eras sobre las 274 estaciones activas en ambas mitades y ARI de bootstrap.

modelo_km2 = KMeans(n_clusters=2, n_init=N_INIT, random_state=SEED).fit(X)
modelo_km4 = KMeans(n_clusters=4, n_init=N_INIT, random_state=SEED).fit(X)

ari_eras_k2 = estabilidad_entre_eras(viajes, estaciones, escalador, modelo_km2, columnas)
ari_boot_k2 = estabilidad_bootstrap(X, modelo_km2.labels_, 2)

ari_eras_k4 = estabilidad_entre_eras(viajes, estaciones, escalador, modelo_km4, columnas)
ari_boot_k4 = estabilidad_bootstrap(X, modelo_km4.labels_, 4)

tabla_estabilidad = pd.DataFrame([
    {"Modelo": "p3-kmeans-k2 (óptimo global)", "k": 2,
     "hist_silhouette": round(silhouette_score(X, modelo_km2.labels_), 3),
     "estabilidad_eras_ari (274 est. comunes)": round(ari_eras_k2, 3),
     "estabilidad_bootstrap_ari (30x 80%)": round(ari_boot_k2, 3)},
    {"Modelo": "p3-kmeans-k4 (operativo granular)", "k": 4,
     "hist_silhouette": round(silhouette_score(X, modelo_km4.labels_), 3),
     "estabilidad_eras_ari (274 est. comunes)": round(ari_eras_k4, 3),
     "estabilidad_bootstrap_ari (30x 80%)": round(ari_boot_k4, 3)},
]).set_index("Modelo")
display(tabla_estabilidad)

,k,hist_silhouette,estabilidad_eras_ari (274 est. comunes),estabilidad_bootstrap_ari (30x 80%)
Modelo,,,,
p3-kmeans-k2 (óptimo global),2,0.151,0.618,0.952
p3-kmeans-k4 (operativo granular),4,0.105,0.571,0.841


---

## 6. Respuesta a la pregunta P3: ¿Qué grupos de estaciones comparten patrones de uso y dónde se ubican en la ciudad?

Presentamos ahora la caracterización cuantitativa y territorial completa de las dos particiones principales registradas en MLflow (**$k = 2$** y **$k = 4$**), cruzando los indicadores de operación con los tres municipios (`GDL`, `ZPN`, `TLQ`) y las estaciones abiertas en 2026.

In [7]:
# Objetivo: describir cuantitativamente y por municipio los grupos de K-Means para k = 2 y k = 4,
# e identificar las estaciones más representativas de cada grupo.
# Resultado esperado: tablas de indicadores promedio por grupo, distribución en GDL/ZPN/TLQ y ejemplos reales.

print("=== PARTICIÓN MACRO: K-MEANS k = 2 ===")
desc_k2 = describir_grupos(matriz, modelo_km2.labels_, estaciones)
display(desc_k2[[
    "estaciones", "nuevas_2026", "est_GDL", "est_ZPN", "est_TLQ",
    "salidas_por_mes", "viajes_por_anclaje_mes", "flujo_neto_por_mes", "balance",
    "pico_manana_07_09", "pico_tarde_17_19", "fin_de_semana", "prop_viajes_redondos",
    "duracion_mediana_min", "meses_activa", "share_viajes_ultimos_12m", "hora_pico"
]])

print("\n=== PARTICIÓN OPERATIVA PARA PLANEACIÓN: K-MEANS k = 4 ===")
desc_k4 = describir_grupos(matriz, modelo_km4.labels_, estaciones)
display(desc_k4[[
    "estaciones", "nuevas_2026", "est_GDL", "est_ZPN", "est_TLQ",
    "salidas_por_mes", "viajes_por_anclaje_mes", "flujo_neto_por_mes", "balance",
    "pico_manana_07_09", "pico_tarde_17_19", "fin_de_semana", "prop_viajes_redondos",
    "duracion_mediana_min", "meses_activa", "share_viajes_ultimos_12m", "hora_pico"
]])

con_info = matriz.join(estaciones[["name", "municipio", "capacidad"]]).assign(grupo_k4=modelo_km4.labels_)
for g in sorted(con_info["grupo_k4"].unique()):
    sub = con_info[con_info["grupo_k4"] == g].sort_values("salidas_por_mes", ascending=False)
    ejemplos_txt = ", ".join(f"{r["name"]} ({int(r["salidas_por_mes"])} sal/mes)" for _, r in sub.head(3).iterrows())
    print(f"Grupo {g} (n={len(sub)}) — Top 3 estaciones por volumen mensual: {ejemplos_txt}")

=== PARTICIÓN MACRO: K-MEANS k = 2 ===


,estaciones,nuevas_2026,est_GDL,est_ZPN,est_TLQ,salidas_por_mes,viajes_por_anclaje_mes,flujo_neto_por_mes,balance,pico_manana_07_09,pico_tarde_17_19,fin_de_semana,prop_viajes_redondos,duracion_mediana_min,meses_activa,share_viajes_ultimos_12m,hora_pico
grupo,,,,,,,,,,,,,,,,,
0,270,20,219,50,1,1060.060,123.817,-1.862,-0.026,0.170,0.260,0.186,0.040,10.376,109.844,0.193,18
1,199,78,133,52,14,322.387,38.616,16.301,-0.008,0.245,0.195,0.240,0.054,9.783,44.824,0.551,8



=== PARTICIÓN OPERATIVA PARA PLANEACIÓN: K-MEANS k = 4 ===


,estaciones,nuevas_2026,est_GDL,est_ZPN,est_TLQ,salidas_por_mes,viajes_por_anclaje_mes,flujo_neto_por_mes,balance,pico_manana_07_09,pico_tarde_17_19,fin_de_semana,prop_viajes_redondos,duracion_mediana_min,meses_activa,share_viajes_ultimos_12m,hora_pico
grupo,,,,,,,,,,,,,,,,,
0,134,0,126,6,2,1460.625,171.186,57.787,0.017,0.178,0.227,0.195,0.045,9.368,126.716,0.116,18
1,132,28,87,45,0,590.000,69.413,-75.688,-0.081,0.149,0.302,0.184,0.036,11.577,85.932,0.328,18
2,149,33,98,39,12,426.841,50.919,40.747,0.012,0.265,0.192,0.208,0.047,9.240,60.872,0.413,8
3,54,37,41,12,1,243.854,26.421,-20.051,-0.037,0.218,0.190,0.307,0.067,10.892,21.944,0.766,8


Grupo 0 (n=134) — Top 3 estaciones por volumen mensual: (GDL-049) Lopez Cotilla / Penitenciaría (5700 sal/mes), (GDL-198) Av. Alcalde / C. Hospital (4858 sal/mes), (GDL-009) Calz. Federalismo/ C. J. Angulo (4099 sal/mes)
Grupo 1 (n=132) — Top 3 estaciones por volumen mensual: (GDL-127) Av. Vallarta / Av. de Los Arcos (3288 sal/mes), (GDL-132) C. Lope de Vega / Av. La Paz (1893 sal/mes), (GDL-093) C. H Colegio Militar / Av. Américas (1703 sal/mes)
Grupo 2 (n=149) — Top 3 estaciones por volumen mensual: (GDL-208) Av. España / Av. Federalismo (1640 sal/mes), (GDL-210) Av. Washington / Calz. Independenci (1410 sal/mes), (GDL-237)Av. Ramón Corona / C. Leandro Valle (1379 sal/mes)
Grupo 3 (n=54) — Top 3 estaciones por volumen mensual: (GDL-044) C. Gral. San Martín /Av.  Vallarta (1053 sal/mes), (GDL-280) Av. Vallarta / C. Aurelio Aceves (1005 sal/mes), (GDL-218) C.Hércules / Av. De la Arboleda (694 sal/mes)


In [8]:
# Objetivo: visualizar el mapa geográfico y las firmas horarias promedio de los 4 grupos operativos (k = 4).
# Resultado esperado: mapa de Guadalajara/Zapopan/Tlaquepaque coloreado por grupo y curvas de 0 a 23 h.

nombres_g4 = {
    0: "G0 · Núcleo hiper-activo receptor (134 est.)",
    1: "G1 · Consolidadas de salida vespertina / empleo (132 est.)",
    2: "G2 · Emisoras matutinas / residenciales (149 est.)",
    3: "G3 · Nuevas periféricas y recreativas de fin de semana (54 est.)",
}
colores_g4 = {0: "#1f77b4", 1: "#ff7f0e", 2: "#2ca02c", 3: "#d62728"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
for g in range(4):
    sub = con_info[con_info["grupo_k4"] == g]
    ax1.scatter(sub["longitud"], sub["latitud"], s=28, alpha=0.85,
                color=colores_g4[g], label=nombres_g4[g])
ax1.set_title("Mapa de los 4 grupos de estaciones por patrón de uso (K-Means k = 4)")
ax1.set_xlabel("Longitud")
ax1.set_ylabel("Latitud")
ax1.legend(fontsize=8, loc="lower left")

horas_cols = [f"h{h:02d}" for h in range(24)]
perfiles_k4 = con_info.groupby("grupo_k4")[horas_cols].mean()
for g, fila in perfiles_k4.iterrows():
    ax2.plot(range(24), fila.to_numpy() * 100, marker="o", ms=3.5,
             color=colores_g4[g], label=nombres_g4[g])
ax2.axvspan(7, 9, color="#2ca02c", alpha=0.10, label="Franja pico mañana (7-9 h)")
ax2.axvspan(17, 19, color="#ff7f0e", alpha=0.10, label="Franja pico tarde (17-19 h)")
ax2.set_title("Firma horaria promedio por grupo (% de salidas en cada hora)")
ax2.set_xlabel("Hora del día")
ax2.set_ylabel("% de salidas del grupo")
ax2.set_xticks(range(0, 24, 2))
ax2.legend(fontsize=7.8)
fig.tight_layout()

### Síntesis cuantitativa de los grupos y decisiones que habilitan para MiBici

| Grupo ($k = 4$) | Estaciones (Nuevas 2026) | Ubicación por municipio | Patrón cuantitativo de uso | Interpretación urbana y decisión para MiBici |
|---|---|---|---|---|
| **Grupo 0 · Núcleo central de máxima rotación y sumidero** | **134** (0 nuevas) | **126 GDL (94.0 %)**, 6 ZPN, 2 TLQ · Polígono central histórico (López Cotilla, Alcalde, Federalismo, Chapultepec) | **1,460.6 salidas/mes** (**181.2 viajes/anclaje-mes**), antigüedad media **127.4 meses**, flujo neto positivo **+12.7 bicis/mes**, viajes cortos (**9.28 min**), alta conectividad (**348 destinos medianos**), uso continuo con doble pico (17.8 % AM y 22.7 % PM). | Son la columna vertebral del sistema. Con **181 viajes por anclaje al mes** operan al límite físico de sus puertos: requieren **ampliación prioritaria de anclajes** o estaciones gemelas a $< 200$ m para absorber llegadas netas. |
| **Grupo 1 · Polos de empleo/comercio con vaciado vespertino** | **132** (28 nuevas) | **87 GDL (65.9 %)** y **45 ZPN (34.1 %)**, 0 TLQ · Corredor poniente y financiero (Av. Vallarta, Los Arcos, La Paz, Américas, Providencia) | **590.0 salidas/mes** (**66.4 viajes/anclaje-mes**), **pico vespertino dominante (30.2 % de todas sus salidas ocurren entre 17 y 19 h** vs. solo 14.9 % entre 7 y 9 h), menor uso en fin de semana (**18.4 %**) y **fuerte flujo neto negativo (−76.1 bicis/mes, balance −0.088)** con viajes más largos (**11.71 min**). | Zonas de oficinas, universidades y comercio donde los usuarios **toman bicicleta al salir por la tarde**. Se vacían sistemáticamente entre 17 y 19 h: necesitan **abastecimiento preventivo de bicicletas antes de las 16:30 h**. |
| **Grupo 2 · Zonas residenciales y alimentadoras matutinas** | **149** (33 nuevas) | **98 GDL (65.8 %)**, **39 ZPN (26.2 %)** y **12 TLQ (8.1 %)** · Anillo intermedio oriente/norte y corredor Tlaquepaque (Av. España, Washington, Río Nilo) | **426.8 salidas/mes** (**50.6 viajes/anclaje-mes**), **pico matutino más alto de toda la red (26.5 % de sus salidas entre 7 y 9 h**, hora pico a las **8:00 h**, contra 19.2 % en la tarde), flujo neto positivo (**+2.6 bicis/mes, balance +0.008**) y viajes de **9.53 min**. | Barrios habitacionales de donde los usuarios **salen entre 7 y 9 h** y a donde regresan al final del día. Requieren **disponibilidad garantizada de bicicletas a las 6:30 h** y anclajes libres por la noche. |
| **Grupo 3 · Expansión periférica y uso recreativo/dominical** | **54** (**37 nuevas de 2026**, 68.5 % del grupo) | **41 GDL (75.9 %)**, **12 ZPN (22.2 %)**, 1 TLQ · Periferia sur/oriente y bordes de expansión 2026 | **243.9 salidas/mes** (**26.7 viajes/anclaje-mes**), antigüedad media **22.3 meses** (mediana **4 meses**, **76.6 %** de sus viajes en los últimos 12 meses), **máximo uso en fin de semana (30.7 % en sábado y domingo)** y **mayor tasa de viajes redondos (`Origen == Destino` = 6.7 %**, 68 % más alta que en el Grupo 1). | Estaciones nuevas en fase de adopción y nodos recreativos de fin de semana (Vía RecreActiva y parques barriales). Necesitan **promoción de conectividad entre semana** y **rebalanceo específico para sábado y domingo**. |